# Week 4 실습 — 수집 코드의 세 가지 모양

| 모양 | 원천 | 이 노트북 |
|---|---|---|
| **한 번 읽기** | 파일 · DB | `yellow_tripdata_2023-01.parquet` → pandas / SQLite + `read_sql` |
| **반복해서 묻기** | API | 국토교통부 수단통행량 OpenAPI — `totalCount`로 끝을 안다 |
| **끝없이 받기** | 스트림 | 업비트 WebSocket 가상자산 실시간 체결 — **위치(offset)** 와 **창(window)** |

> 준비: `pip install -r requirements.txt` → 이 노트북을 위에서부터 순서대로 실행

뉴욕시 택시·리무진 위원회(NYC TLC)의 공개 데이터 "TLC Trip Record Data"**에서 가져온 파일입니다.
- 공식 페이지: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
- 파일 직접 주소 (README.md:27의 다운로드 명령):



In [15]:
# 셀 0 — 공통 준비
import os, json, time, sqlite3, threading, queue
from pathlib import Path
import pandas as pd
import requests

for d in ["data", "raw", "raw/api", "state"]:
    Path(d).mkdir(parents=True, exist_ok=True)

PARQUET = "yellow_tripdata_2023-01.parquet"
DB_PATH = "data/taxi.db"
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
print("준비 완료")

준비 완료


---
## PART 1-A · 한 번 읽기 — 파일

받으면 끝. 코드는 한 번 실행된다.
파일 수집의 어려움은 읽기가 아니라 **"빠진 날이 없는가"** 를 확인하는 것.

Apache Parquet(아파치 파케이)는 대규모 데이터 분석 및 처리 환경에서 널리 사용되는 오픈소스 컬럼(Column, 열) 기반 데이터 저장 파일 포맷입니다.

흔히 사용하는 CSV나 JSON과 같은 파일이 데이터를 행(Row) 단위로 저장하는 것과 달리, Parquet는 데이터를 열(Column) 단위로 모아서 저장하는 것이 가장 큰 특징

![Parquet](Parquet.png)

In [16]:
# 셀 1 — 파일 통째로 읽기
trips = pd.read_parquet(PARQUET)
print(trips.shape)
print(round(trips.memory_usage(deep=True).sum() / 1024**2), "MB")
trips.head(3)

(3066766, 19)
448 MB


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0


### 컬럼 설명 — 한 행 = 택시 운행 1건 (19개 열)

미터기(TPEP 단말기)가 운행이 끝날 때마다 기록한 값이다. 이 열 이름이 그대로 DB 테이블(`trips`)의 열 이름이 되므로, 뒤의 SQL을 쓸 때 이 표를 참고한다.

**언제 · 어디서**

| 컬럼 | 타입 | 의미 |
|---|---|---|
| `VendorID` | int | 기록을 보낸 미터기 업체 (1 = Creative Mobile Technologies, 2 = VeriFone) |
| `tpep_pickup_datetime` | datetime | 승차 시각 (미터기를 켠 시각) — 이 노트북에서 날짜 조건·인덱스·증분 수집의 기준 |
| `tpep_dropoff_datetime` | datetime | 하차 시각 (미터기를 끈 시각) |
| `PULocationID` | int | 승차 구역 번호 (PU = Pick-Up). TLC가 뉴욕을 나눈 택시 구역 1~265 |
| `DOLocationID` | int | 하차 구역 번호 (DO = Drop-Off) |

**운행 내용**

| 컬럼 | 타입 | 의미 |
|---|---|---|
| `passenger_count` | float | 승객 수 — 기사가 직접 입력하는 값이라 0이나 빈 값이 있다 |
| `trip_distance` | float | 운행 거리, 단위는 **마일** (1마일 ≈ 1.6 km) |
| `RatecodeID` | float | 요금제: 1 = 일반, 2 = JFK 공항 정액, 3 = 뉴어크 공항, 4 = 나소/웨스트체스터, 5 = 협의 요금, 6 = 합승, 99 = 알 수 없음 |
| `store_and_fwd_flag` | str | Y = 통신이 끊겨 단말기에 저장해 두었다가 나중에 전송, N = 실시간 전송 |
| `payment_type` | int | 결제수단: 1 = 카드, 2 = 현금, 3 = 무료, 4 = 분쟁, 0 = 기록 없음 |

**요금 (달러)**

| 컬럼 | 타입 | 의미 |
|---|---|---|
| `fare_amount` | float | 미터기 기본 요금 (시간·거리로 계산된 금액) |
| `extra` | float | 할증 — 심야·출퇴근 시간대 추가 요금 등 |
| `mta_tax` | float | MTA(뉴욕 교통공사) 세금 0.5달러 |
| `tip_amount` | float | 팁 — **카드 결제분만** 자동 기록된다. 현금 팁은 0으로 남는다 |
| `tolls_amount` | float | 통행료 (다리·터널) |
| `improvement_surcharge` | float | 택시 개선 부담금 (운행당 1달러, 이전에는 0.3달러) |
| `congestion_surcharge` | float | 맨해튼 혼잡 통행료 2.5달러 |
| `airport_fee` | float | 공항 승차 요금 1.25달러 (라과디아·JFK에서 탈 때) |
| `total_amount` | float | 승객에게 청구된 합계 — 위 항목들의 합. 현금 팁은 들어 있지 않다 |

> **수집하는 사람이 눈여겨볼 점** (아래 셀로 직접 확인)
> - 정수일 것 같은 `passenger_count`·`RatecodeID`가 **float**다 — 값이 빈 행이 있는 열이라 실수형으로 저장되어 있다.
> - 71,743행은 `passenger_count`·`RatecodeID`·`store_and_fwd_flag`·`congestion_surcharge`·`airport_fee`가 **한꺼번에 비어 있고**, 그 행들의 `payment_type`은 모두 0이다.
> - 요금 열에는 **음수**도 있다 (환불·분쟁 기록). 수집 단계에서는 지우지 않고 그대로 둔다.
> - 구역 번호를 이름으로 바꾸려면 TLC의 `taxi_zone_lookup.csv`(Taxi Zone Lookup Table)와 조인한다.

In [ ]:
# 셀 1-1 — 열 이름 · 타입 · 빈 값 개수를 한 표로
info = pd.DataFrame({"dtype": trips.dtypes.astype(str),      # 열의 자료형
                     "nulls": trips.isna().sum(),            # 빈 값(NaN) 개수
                     "example": trips.iloc[0]})              # 첫 행의 값 (예시)
print(info)
# 빈 값이 있는 행들의 payment_type 은? → 전부 0
print(trips.loc[trips["passenger_count"].isna(), "payment_type"].value_counts())

In [6]:
# 셀 2 — 파일 이름은 '2023-01'인데, 정말 1월만 들어 있나? 빠진 날은?
day = trips["tpep_pickup_datetime"].dt.date
counts = day.value_counts().sort_index()

in_jan = counts[(counts.index >= pd.Timestamp("2023-01-01").date()) &
                (counts.index <= pd.Timestamp("2023-01-31").date())]
print("1월 날짜 수:", len(in_jan), "/ 31")
print("1월 밖 날짜 (섞여 들어온 행):")
print(counts[~counts.index.isin(in_jan.index)])

1월 날짜 수: 31 / 31
1월 밖 날짜 (섞여 들어온 행):
tpep_pickup_datetime
2008-12-31     2
2022-10-24     4
2022-10-25     7
2022-12-31    25
2023-02-01    10
Name: count, dtype: int64


---
## PART 1-B · 한 번 읽기 — DB (SQL 날려 보기)

파일에는 "골라 달라"고 말할 상대가 없다. DB에는 질문(SQL)을 받아 **답만** 보내 주는 쪽이 있다.
실습용으로 parquet를 SQLite 파일 하나(`data/taxi.db`)에 넣고, 그다음부터는 SQL로만 꺼낸다.

오늘 쓰는 SQL 세 단어: `SELECT` 어느 열을 · `WHERE` 어느 행을 · `GROUP BY` 무엇끼리 묶어서

In [ ]:
# 셀 3 — parquet → SQLite 적재 (처음 한 번만, 30초~1분)
con = sqlite3.connect(DB_PATH)

exists = con.execute(
    "SELECT count(*) FROM sqlite_master WHERE type='table' AND name='trips'").fetchone()[0]
# trips 테이블이 아직 없을 때만 적재 (있으면 0이 아니라서 건너뜀 → 셀을 다시 실행해도 중복 적재 안 됨)
if not exists:
    # 시작 시각 기록 — 적재에 몇 초 걸렸는지 재려고
    t0 = time.time()
    # DataFrame을 'trips' 테이블로 저장. index=False: 0,1,2… 행 번호는 열로 넣지 않음
    # chunksize=200_000: 307만 행을 20만 행씩 나눠 INSERT (한 번에 넣으면 메모리 부담)
    trips.to_sql("trips", con, index=False, chunksize=200_000)
    # 승차 시각 열에 인덱스(색인) 생성 — WHERE tpep_pickup_datetime >= ... 같은 날짜 조건 검색이 빨라짐
    con.execute("CREATE INDEX idx_pickup ON trips(tpep_pickup_datetime)")
    # 변경 내용을 DB 파일(data/taxi.db)에 확정 저장
    con.commit()
    print(f"적재 완료: {time.time() - t0:.1f}초")
print(con.execute("SELECT count(*) FROM trips").fetchone()[0], "행이 DB에 있음")

In [ ]:
# 셀 4 — read_sql 한 줄: 쿼리를 보내고 DataFrame으로 받는다 (LIMIT 5 = SQL의 head())
df = pd.read_sql("SELECT * FROM trips LIMIT 5", con)
df

In [ ]:
# 셀 5 — WHERE / GROUP BY 한 줄이 '돌아오는 양'을 정한다. 시간이 더 오래 걸리는 것은? 이유는
q_all = """SELECT * FROM trips
           WHERE tpep_pickup_datetime >= '2023-01-31'
             AND tpep_pickup_datetime <  '2023-02-01'"""

q_one = """SELECT payment_type,
                  COUNT(*)                    AS trips,
                  ROUND(SUM(total_amount), 0) AS sales
           FROM trips
           WHERE tpep_pickup_datetime >= '2023-01-31'
             AND tpep_pickup_datetime <  '2023-02-01'
           GROUP BY payment_type
           ORDER BY trips DESC"""

a = pd.read_sql(q_all, con)
b = pd.read_sql(q_one, con)
print(a.shape, a.memory_usage(deep=True).sum() // 1024, "KB")
print(b.shape, b.memory_usage(deep=True).sum() // 1024, "KB")
a

(100372, 19) 18731 KB
(5, 3) 0 KB


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-31 00:00:02,2023-01-31 00:08:20,1.0,1.93,1.0,N,263,238,1,11.4,1.0,0.5,2.00,0.0,1.0,18.40,2.5,0.00
1,2,2023-01-31 00:00:06,2023-01-31 00:18:36,1.0,6.79,1.0,N,132,134,1,31.0,1.0,0.5,6.70,0.0,1.0,41.45,0.0,1.25
2,2,2023-01-31 00:00:09,2023-01-31 00:30:40,1.0,11.98,1.0,N,186,188,1,49.9,1.0,0.5,6.10,0.0,1.0,61.00,2.5,0.00
3,2,2023-01-31 00:00:15,2023-01-31 00:06:31,1.0,1.93,1.0,N,90,48,2,10.0,1.0,0.5,0.00,0.0,1.0,15.00,2.5,0.00
4,2,2023-01-31 00:00:16,2023-01-31 00:09:28,1.0,2.78,1.0,N,249,246,1,13.5,1.0,0.5,3.70,0.0,1.0,22.20,2.5,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100367,2,2023-01-31 23:59:53,2023-02-01 00:02:02,1.0,0.91,1.0,N,141,263,1,5.8,1.0,0.5,1.00,0.0,1.0,11.80,2.5,0.00
100368,2,2023-01-31 23:59:54,2023-02-01 00:10:14,1.0,2.61,1.0,N,263,100,1,14.2,1.0,0.5,2.00,0.0,1.0,21.20,2.5,0.00
100369,2,2023-01-31 23:59:57,2023-02-01 00:06:29,1.0,3.04,1.0,N,132,216,2,13.5,1.0,0.5,0.00,0.0,1.0,17.25,0.0,1.25
100370,2,2023-01-31 23:59:59,2023-02-01 00:09:31,1.0,2.69,1.0,N,90,163,1,13.5,1.0,0.5,3.70,0.0,1.0,22.20,2.5,0.00


### 직접 SQL 날려 보기

아래 셀의 `q`만 바꿔 가며 실행해 보세요. 열 이름은 셀 4 출력 참고.
`payment_type`: 1=카드, 2=현금, 3=무료, 4=분쟁 · `PULocationID`/`DOLocationID`: 승차/하차 구역 번호

In [ ]:
# 셀 6 — 시간대별 운행 수와 평균 요금
# avg_total, avg_miles 만들기, total_amount, trip_distance 로
# '2023-01-01' '2023-02-01' 사이 tpep_pickup_datetime 사용
q = """
SELECT strftime('%H', tpep_pickup_datetime) AS hour,
       COUNT(*)                            AS trips,
       ROUND(AVG(total_amount), 2)         AS avg_total,
       ROUND(AVG(trip_distance), 2)        AS avg_miles
FROM trips
WHERE 
_____________
ORDER BY ____
"""
pd.read_sql(q, con)

,hour,trips,avg_total,avg_miles
0,00,84957,28.44,4.02
1,01,59799,26.06,3.49
2,02,42040,24.66,3.20
3,03,27437,25.73,3.74
4,04,17835,30.94,4.77
5,05,18011,36.06,15.30
6,06,43860,30.23,5.42
7,07,86876,26.68,5.25
8,08,116865,25.04,5.52
9,09,131110,25.29,3.12


In [ ]:
# 셀 7 — 승차가 가장 많은 구역 TOP 10 + 카드 결제 팁 비율 팁/결제금액 비율
# payment_type = 1 카드 결제만, 현금은 기록되지 않기 때문
# tip_amount(팁), fare_amount(요금), PULocationID(지역 아이디)
q = """
SELECT PULocationID,
       COUNT(*) AS trips,
       ROUND(100.0 * ________________, 1) AS tip_pct
FROM trips
WHERE payment_type = 1 AND ____?____ > 0
GROUP BY PULocationID
ORDER BY trips DESC
LIMIT ___
"""
pd.read_sql(q, con)

,PULocationID,trips,tip_pct
0,237,119111,25.0
1,236,112944,24.3
2,132,109380,18.4
3,161,108366,24.3
4,186,85945,23.6
5,162,85292,24.1
6,142,80690,24.6
7,230,74452,23.7
8,138,74342,23.4
9,170,71085,23.6


**연습 문제** — `q`를 직접 써 보세요
- `trip_distance`가 0 이하이거나 `total_amount`가 음수인 '이상한 행'은 몇 개?
- 공항 요금(`airport_fee > 0`)이 붙은 운행의 평균 `total_amount`는?

In [ ]:
# 셀 8 — 매일 전부 다시 받지 않는다: 증분(incremental) 수집
#   이 셀을 여러 번 실행해 보세요. 실행할 때마다 '지난번 이후 하루치'만 가져온다.
STATE = Path("state/taxi_last_ts.txt")
last_ts = STATE.read_text().strip() if STATE.exists() else "2022-12-31 23:59:59"  # 지난번 끝
until   = (pd.Timestamp(last_ts).normalize() + pd.Timedelta(days=1, hours=23, minutes=59, seconds=59))

new = pd.read_sql(
    """SELECT * FROM trips
       WHERE tpep_pickup_datetime > ? AND tpep_pickup_datetime <= ?
       ORDER BY tpep_pickup_datetime""",
    con, params=(last_ts, str(until)))

if len(new):                                             # 0행이면 위치 그대로
    out = f"raw/trips_after_{last_ts[:10]}.csv"
    new.to_csv(out, index=False)                         # 받은 그대로 저장
    STATE.write_text(str(new["tpep_pickup_datetime"].max()))  # 저장이 끝난 '후'에 위치 기록
    print(len(new), "행만 전송 →", out)
print("다음 시작점:", STATE.read_text() if STATE.exists() else last_ts)

> 처음부터 다시 하고 싶으면 `state/taxi_last_ts.txt`를 지우면 된다.
> 이 **"기억하는 값"** 이 PART 3에서 **위치(offset)** 라는 이름으로 다시 나온다.

---
## PART 1-C · 반복해서 묻기 — API (국토교통부 수단통행량)

끝이 있는 반복. **"페이지가 더 있나?"를 서버(totalCount)에 묻는다.**

| 오퍼레이션 | 날짜 파라미터 |
|---|---|
| `getDailyTransportationModeTripVolume` (일별) | `opr_ymd=20250801` |
| `getMonthlyTransportationModeTripVolume` (월별) | `opr_ym=202508` |
| `getAnnualTransportationModeTripVolume` (연별) | `opr_yr=2025` |
| `getTransportationModeTripVolumeforPeoplewithReducedMobility` (교통약자) | `opr_ymd=20250801` |

공통: `ctpv_cd`(시도코드, 예: 29 광주), `sgg_cd`(시군구코드, 예: 29140 광주 서구), `numOfRows` 최대 1000

In [ ]:
# 셀 9 — 설정과 fetch 함수
BASE_URL = "https://apis.data.go.kr/1613000/TransportationModeTripVolume"

def load_key(name="DATA_GO_KR_KEY"):
    """키는 코드에 쓰지 않는다 — 환경변수, 없으면 .env 파일(깃에 올리지 않음)에서 읽는다."""
    if os.getenv(name):
        return os.environ[name]
    if Path(".env").exists():
        for line in Path(".env").read_text(encoding="utf-8").splitlines():
            k, _, v = line.partition("=")
            if k.strip() == name and v.strip():
                return v.strip().strip("\"'")
    raise RuntimeError(f"{name} 가 없습니다 — .env.example 을 .env 로 복사하고 키를 넣으세요")

SERVICE_KEY = load_key()

def fetch(operation, params):
    """한 페이지를 요청해서 dict로 돌려준다."""
    p = {"serviceKey": SERVICE_KEY, "dataType": "JSON", **params}
    r = requests.get(f"{BASE_URL}/{operation}", params=p, timeout=30)
    try:
        return r.json()           # 에러도 JSON 상자로 오는 경우가 많다 → check()에서 판단
    except ValueError:
        r.raise_for_status()      # JSON이 아니면 HTTP 에러부터 확인
        raise

OP = "getDailyTransportationModeTripVolume"
QUERY = {"opr_ymd": "20250801", "ctpv_cd": "29", "sgg_cd": "29140"}   # 2025-08-01 광주 서구

data = fetch(OP, {**QUERY, "pageNo": 1, "numOfRows": 3})
Path("raw/api/sample.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(data, ensure_ascii=False, indent=2)[:900])   # 상자를 열기 전에 구조부터

**생각해 보기** — `status_code`는 200인데 표가 비어 있다면? → HTTP 상태만 보면 안 되고 **응답 안의 header와 body**를 봐야 한다.

- 정상: `Response.header.resultCode == "200"`
- 페이지 번호가 끝을 넘으면: 200 · SUCCESS 인데 `item`이 빈 리스트
- 키가 틀리면: 아예 다른 상자(`OpenAPI_ServiceResponse`)가 온다

In [ ]:
# 셀 10 — 응답 검사: 상자의 이름부터 읽는다
def check(data):
    if "OpenAPI_ServiceResponse" in data:                  # 인증키 오류 등 게이트웨이 에러
        h = data["OpenAPI_ServiceResponse"]["cmmMsgHeader"]
        raise RuntimeError(f"[{h['returnReasonCode']}] {h['errMsg']} ({h['returnAuthMsg']})")
    header = data["Response"]["header"]
    if header["resultCode"] not in ("00", "200"):
        raise RuntimeError(f"[{header['resultCode']}] {header['resultMsg']}")
    return data["Response"]["body"]

# ① 끝을 넘은 페이지 — status 200, SUCCESS, 그런데 표가 비어 있다
r = requests.get(f"{BASE_URL}/{OP}", timeout=30,
                 params={"serviceKey": SERVICE_KEY, "dataType": "JSON", **QUERY,
                         "pageNo": 99, "numOfRows": 1000})
empty = check(r.json())
print("① status_code:", r.status_code, "/ items:", empty["items"]["item"], "/ totalCount:", empty["totalCount"])

# ② 틀린 키 — 상자 이름부터 다르다
r = requests.get(f"{BASE_URL}/{OP}", timeout=30,
                 params={"serviceKey": "WRONG_KEY", "dataType": "JSON", **QUERY})
print("② status_code:", r.status_code)
try:
    check(r.json())
except RuntimeError as e:
    print("   잡았다 →", e)

body = check(data)
print("정상 → totalCount:", body["totalCount"], "/ pageNo:", body["pageNo"], "/ numOfRows:", body["numOfRows"])

In [ ]:
# 셀 11 — 상자를 세 번 열면 json_normalize 한 줄
items = data["Response"]["body"]["items"]["item"]      # 상자: Response → body → items → item
df = pd.json_normalize(items)
print(df.head(3))
print(df.dtypes)          # 코드 값('01', '05')은 문자, 인원수는 숫자로 온다 — 앞자리 0을 지키려면 문자 그대로!

In [ ]:
# 셀 12 — 여러 페이지: 끝은 코드에 박지 않고 서버(totalCount)에게 묻는다
def fetch_all(operation, query, num_rows=1000, save_prefix=None):
    rows, page = [], 1
    while True:                                           # 끝을 모른다
        body = check(fetch(operation, {**query, "pageNo": page, "numOfRows": num_rows}))
        item = body["items"]["item"] if body["items"] else []
        if isinstance(item, dict):                        # 1건이면 리스트가 아니라 dict로 오는 API도 있다
            item = [item]
        if save_prefix:                                   # raw/ 에 받은 그대로 저장
            Path(f"{save_prefix}_p{page}.json").write_text(
                json.dumps(body, ensure_ascii=False), encoding="utf-8")
        rows += item
        print(f"  page {page}: {len(item)}건 (누적 {len(rows)} / {body['totalCount']})")
        if not item or page * num_rows >= body["totalCount"]:   # 끝
            break
        page += 1
        time.sleep(0.3)                                   # 서버 예의
    return pd.DataFrame(rows), body["totalCount"]

daily, total = fetch_all(OP, QUERY, save_prefix="raw/api/daily_20250801_29140")
print(len(daily), "건 /", total, "건")

In [ ]:
# 셀 13 — 받은 표로 간단히: 교통수단 × 시간대별 발생 통행 인원
print(daily.groupby("trfc_mns_nm")["ocrn_pasg_nope"].sum().sort_values(ascending=False))
print()
print(daily.groupby("users_type_nm")["ocrn_pasg_nope"].sum().sort_values(ascending=False))
daily.pivot_table(index="tzon", columns="trfc_mns_nm", values="ocrn_pasg_nope", aggfunc="sum").fillna(0).astype(int)

**연습 문제**
1. `OP`를 월별(`getMonthlyTransportationModeTripVolume`)로 바꾸고 `QUERY`를 `{"opr_ym": "202508", ...}`로 — 몇 페이지가 나오나?
2. `numOfRows`를 100으로 줄이면 페이지 수는? 1001로 하면? (가이드: 최대 1000)
3. 다른 날짜/지역으로 받아서 `raw/api/`에 쌓아 보기. 받은 파일 수와 `totalCount`로 빠진 페이지가 없는지 확인.

---
## PART 3 · 끝없이 받기 — 가상자산 실시간 시세 (업비트 WebSocket)

끝이 없는 반복. **"어디까지 받았나"를 반드시 기억한다.**

- 원천: `wss://api.upbit.com/websocket/v1` — 인증키 없이 실시간 체결(trade) 수신
- **생산자** (웹소켓 수신) → **중개자** (`queue.Queue` / 덧붙이기만 하는 장부 `raw/upbit_trades.jsonl`) → **소비자** (집계)
- 소비자는 **위치(offset)** 를 `state/`에 적어 두고, 죽었다 살아나도 그 자리부터 읽는다
- 끝없는 흐름을 집계하려면 **창(window)** 을 자른다

> 터미널에서 진짜로 '끝없이' 돌려 보려면 `upbit_producer.py` / `upbit_consumer.py` (README 참고)

In [ ]:
# 셀 14 — 첫 메시지 구조부터 본다
import uuid, certifi, websocket   # websocket-client

WS_URL = "wss://api.upbit.com/websocket/v1"
CODES = ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL", "KRW-DOGE"]
SSLOPT = {"ca_certs": certifi.where()}   # macOS python.org 설치본의 인증서 문제 방지

def subscribe(codes=CODES):
    ws = websocket.create_connection(WS_URL, timeout=10, sslopt=SSLOPT)
    ws.send(json.dumps([{"ticket": str(uuid.uuid4())},
                        {"type": "trade", "codes": codes, "is_only_realtime": True},
                        {"format": "DEFAULT"}]))
    return ws

ws = subscribe()
msg = json.loads(ws.recv())     # 업비트는 bytes로 보낸다 — json.loads가 알아서 처리
ws.close()
print(json.dumps(msg, ensure_ascii=False, indent=2))

주요 필드: `code` 종목 · `trade_price` 체결가 · `trade_volume` 체결량 · `ask_bid` 매도/매수 · `trade_timestamp` 체결 시각(ms) · `sequential_id` 체결 고유번호(중복 제거용)

In [ ]:
# 셀 15 — 생산자 → queue.Queue → 소비자: 처리가 느리면 큐가 쌓인다
q = queue.Queue()                       # 중개자(broker) 역할
stop = threading.Event()

def producer():                         # 생산자: 받는 대로 put 만 한다
    ws = subscribe()
    while not stop.is_set():
        try:
            q.put(json.loads(ws.recv()))
        except websocket.WebSocketTimeoutException:
            continue
    ws.close()

def consumer(per_sec=2):                # 소비자: 초당 2건만 처리할 수 있다고 치자
    while not stop.is_set():
        try:
            q.get(timeout=1)
        except queue.Empty:
            continue
        time.sleep(1 / per_sec)         # 처리에 걸리는 시간 → 이 숫자를 바꿔 보기
        q.task_done()

threading.Thread(target=producer, daemon=True).start()
threading.Thread(target=consumer, daemon=True).start()
for t in range(15):
    time.sleep(1)
    print(f"{t+1:2d}초  큐에서 기다리는 메시지: {q.qsize()}개")
stop.set()

실제 체결 속도는 일정하지 않다 — 장이 뜨거울 땐 큐가 급격히 늘고, 조용할 땐 소비자가 따라잡는다.
`per_sec`를 20으로 올려 보거나, 소비자 스레드를 하나 더 붙여 보자.

`queue.Queue`는 **꺼내면 사라진다.** 그래서 두 번째 소비자가 같은 데이터를 다시 읽을 수 없다.
→ 다음 셀부터는 **덧붙이기만 하는 장부**(jsonl 파일)를 중개자로 쓴다.

In [ ]:
# 셀 16 — 생산자를 백그라운드로: 장부(raw/upbit_trades.jsonl)에 덧붙이기만 한다
LEDGER = Path("raw/upbit_trades.jsonl")
ledger_stop = threading.Event()

def ledger_producer(seconds=120):
    ws, end = subscribe(), time.time() + seconds
    with LEDGER.open("a", encoding="utf-8") as f:
        while not ledger_stop.is_set() and time.time() < end:
            try:
                m = json.loads(ws.recv())
            except websocket.WebSocketTimeoutException:
                continue
            f.write(json.dumps(m, ensure_ascii=False) + "\n")   # 한 줄 = 한 메시지
            f.flush()
    ws.close()

LEDGER.touch()
ledger_stop.clear()
threading.Thread(target=ledger_producer, args=(120,), daemon=True).start()
time.sleep(5)
print("2분 동안 장부에 쌓는 중 — 아래 셀들을 몇 초 간격으로 여러 번 실행해 보세요")

In [ ]:
# 셀 17 — 소비자 그룹마다 자기 위치: seek(위치) → 새 줄만 읽고 → 처리 후 tell()을 기록
def consume(group, max_lines=None):
    pos_file = Path(f"state/upbit_pos_{group}.txt")
    pos = int(pos_file.read_text()) if pos_file.exists() else 0      # 지난번 위치(바이트)
    rows = []
    with LEDGER.open("rb") as f:
        f.seek(pos)                                                  # 그 자리로 점프
        while max_lines is None or len(rows) < max_lines:
            line = f.readline()
            if not line.endswith(b"\n"):                             # 쓰는 중인 반쪽 줄은 다음에
                break
            rows.append(json.loads(line))
            pos = f.tell()
    # --- 여기서 처리(집계·저장)를 끝낸 '후'에 위치를 적는다 ---
    pos_file.write_text(str(pos))
    return pd.DataFrame(rows), pos

new_a, pos_a = consume("A")
print(f"[A] 새로 {len(new_a)}건, 위치 {pos_a} 바이트 / 장부 크기 {LEDGER.stat().st_size}")
if len(new_a):
    print(new_a.groupby("code")["trade_volume"].agg(["count", "sum"]))

In [ ]:
# 셀 18 — 그룹 B는 A와 상관없이 자기 위치에서, 한 번에 5건씩만
new_b, pos_b = consume("B", max_lines=5)
print(f"[B] 새로 {len(new_b)}건, 위치 {pos_b}")
print(new_b[["code", "trade_price", "trade_volume", "ask_bid", "sequential_id"]] if len(new_b) else "없음")
print({p.stem: int(p.read_text()) for p in Path("state").glob("upbit_pos_*.txt")})

- A를 여러 번 실행하면 새로 쌓인 것만 읽고, B는 따로 자기 속도로 따라온다 — **장부는 그대로, 위치는 읽는 쪽에 있다.**
- 처음부터 다시 읽으려면 `state/upbit_pos_A.txt`를 지우면 된다 (새 그룹 C = 0부터).
- 위치를 처리 **전**에 적으면 죽었을 때 잃어버리고, **후**에 적으면 두 번 처리될 수 있다 → `sequential_id`로 중복 제거.

In [ ]:
# 셀 19 — 창(window): 끝없는 흐름을 10초 단위로 잘라 종목별 VWAP·거래량 집계
#   max_seconds 동안 '끝없이' 받다가 멈춘다. 중간에 멈추려면 ■(interrupt)
WINDOW_SEC = 10

def window_stream(max_seconds=40, codes=CODES):
    ws, end = subscribe(codes), time.time() + max_seconds
    window, cur, seen = {}, None, set()
    try:
        while time.time() < end:                                   # 사실상 while True
            try:
                m = json.loads(ws.recv())
            except websocket.WebSocketTimeoutException:
                continue
            if m["sequential_id"] in seen:                         # 같은 체결이 두 번 오면 무시
                continue
            seen.add(m["sequential_id"])
            w = m["trade_timestamp"] // 1000 // WINDOW_SEC * WINDOW_SEC   # 창의 시작 시각(초)
            if cur is not None and w > cur:                        # 다음 창 시작 → 이전 창들 닫기
                for done in sorted(k for k in window if k < w):
                    closed = pd.DataFrame(window.pop(done)).T.sort_index()
                    closed["vwap"] = (closed["amount"] / closed["volume"]).round(0)
                    closed["trades"] = closed["trades"].astype(int)
                    print(f"\n[{pd.to_datetime(done, unit='s', utc=True).tz_convert('Asia/Seoul'):%H:%M:%S}] 창 닫힘")
                    print(closed[["trades", "volume", "vwap", "last"]])
            cur = w if cur is None else max(cur, w)                # 늦게 온 체결은 창을 되돌리지 않는다
            s = window.setdefault(w, {}).setdefault(
                m["code"], {"trades": 0, "volume": 0.0, "amount": 0.0, "last": 0.0})
            s["trades"] += 1
            s["volume"] += m["trade_volume"]
            s["amount"] += m["trade_price"] * m["trade_volume"]
            s["last"] = m["trade_price"]
    except KeyboardInterrupt:
        print("중단")
    finally:
        ws.close()
    print(f"\n(마지막 창은 아직 열려 있음 — {len(window.get(cur, {}))}개 종목 집계 중)")

window_stream(max_seconds=40)

**연습 문제**
1. `WINDOW_SEC`를 60으로 — 배치(`groupby` + `dt.floor("1min")`)로 셀 17의 장부를 집계한 결과와 비교해 보기
2. 창 안에서 `ask_bid == "BID"`(매수) 비율도 같이 내보내기
3. 창이 닫힐 때 결과를 `raw/upbit_windows.jsonl`에 덧붙이기 — 스트림 결과도 결국 파일(배치)이 된다

In [ ]:
# 셀 20 — 정리
ledger_stop.set()
con.close()
print("장부:", LEDGER.stat().st_size if LEDGER.exists() else 0, "바이트")